# Population Analysis, Education and Social Development Planning, Literacy Classification, and District Infrastructure Priority Scoring Using Census 2011 Data
**SI Lab project - Ramdeobaba University, Nagpur**

## 1. Project Overview
**Problem.** District planners need a transparent way to screen districts for further assessment of population pressure, education and social development, and planning priority.

**Objectives.** (A) describe population patterns; (B) analyse literacy and social indicators statistically; (C) classify districts into Low/Medium/High literacy with Random Forest; (D) build a transparent weighted priority score (UIPS) with sensitivity analysis.

**Data.** Only the IndiaSet Census 2011 district dataset (`indiaset/census-2011`, Hugging Face) is used. No external data, no synthetic data, no invented columns.

**Methods.** Descriptive statistics, Pearson/Spearman correlation, Welch t / Mann-Whitney test, one-way ANOVA with assumption checks (Kruskal-Wallis fallback), Random Forest classification against a majority-class baseline, min-max normalised weighted scoring.

**Important.** Census 2011 is historical. Demographic indicators do **not** establish current infrastructure shortages, school capacity, hospital capacity or housing deficits. UIPS is a screening tool only.

All logic lives in the project modules (`data_loader.py`, `preprocessing.py`, `module_a.py` ... `module_d.py`, `visualizations.py`, `statistical_tests.py`) so this notebook and the Streamlit app share identical definitions. Run cells top to bottom.

## 2. Environment Setup and Data Loading

In [1]:
# Cell 1 - install dependencies (skip if already installed)
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
# Cell 2 - imports
import sys, warnings, os
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from scipy import stats
import data_loader, preprocessing as pp, module_a as A, module_b as B, module_c as C, module_d as D
import statistical_tests as ST, visualizations as V
pd.set_option("display.max_columns", 50, "display.width", 200)
warnings.filterwarnings("ignore")
SEED = 42

In [3]:
# Cell 3 - inspect the repository file list BEFORE choosing a filename, then download
try:
    files = data_loader.list_repo_files()
    print("Repository files:", files)
    print("Selected file:", data_loader.choose_data_file(files))
except Exception as e:
    print("Could not list repository (offline?):", e)

Repository files: ['.gitattributes', 'README.md', 'census_2011_districts_final.parquet']
Selected file: census_2011_districts_final.parquet


In [4]:
# Cell 4 - load (download, with local ./data fallback for Parquet/CSV/Excel)
raw, source = data_loader.load_raw()
print("Loaded from:", source)

Loaded from: Hugging Face: indiaset/census-2011/census_2011_districts_final.parquet


In [5]:
# Cell 5 - first rows and shape
display(raw.head())
print("Shape:", raw.shape)

,state_code,state_name,district_code,district_name,lgd_code,district_name_lgd,pop_total,pop_male,pop_female,pop_under6_total,pop_under6_male,pop_under6_female,pop_sc,pop_st,literate_total,literate_male,literate_female,illiterate_total,illiterate_male,illiterate_female,workers_total,workers_male,workers_female,non_workers_total,non_workers_male,non_workers_female,literacy_rate,sex_ratio,workforce_participation
0,1,Jammu & Kashmir,1,Kupwara,8,Kupwara,870354,474190,396164,188798,100459,88339,1048,70352,439654,282823,156831,430700,191367,239333,229064,190899,38165,641290,283291,357999,50.51,835,26.32
1,1,Jammu & Kashmir,2,Badgam,2,Budgam,753745,398041,355704,155202,84720,70482,368,23912,335649,207741,127908,418096,190300,227796,214866,162578,52288,538879,235463,303416,44.53,894,28.51
2,1,Jammu & Kashmir,3,Leh(Ladakh),9,Leh Ladakh,133487,78971,54516,12016,6174,5842,488,95857,93770,62834,30936,39717,16137,23580,75079,53265,21814,58408,25706,32702,70.25,690,56.24
3,1,Jammu & Kashmir,4,Kargil,6,Kargil,140802,77785,63017,19928,10078,9850,18,122336,86236,56301,29935,54566,21484,33082,51873,39839,12034,88929,37946,50983,61.25,810,36.84
4,1,Jammu & Kashmir,5,Punch,10,Poonch,476835,251899,224936,84674,44733,39941,556,176101,261724,163333,98391,215111,88566,126545,161393,117677,43716,315442,134222,181220,54.89,893,33.85


Shape: (640, 29)


In [6]:
# Cell 6 - column names and data types
display(raw.dtypes.to_frame("dtype"))

,dtype
state_code,int64
state_name,str
district_code,int64
district_name,str
lgd_code,int64
district_name_lgd,str
pop_total,int64
pop_male,int64
pop_female,int64
pop_under6_total,int64


In [7]:
# Cell 7 - missing values and duplicates
print("Missing values per column (non-zero only):"); display(raw.isna().sum()[lambda s: s > 0])
print("Missing total:", int(raw.isna().sum().sum()))
print("Duplicate rows:", raw.duplicated().sum(), "| duplicate district_code:", raw["district_code"].duplicated().sum())

Missing values per column (non-zero only):


Series([], dtype: int64)

Missing total: 0
Duplicate rows: 0 | duplicate district_code: 0


In [8]:
# Cell 8 - data dictionary and variable mapping (source -> conceptual)
display(pp.data_dictionary(raw)); display(pp.schema_report(raw))

,Source column,Dtype,Mapped concept
0,state_code,int64,State_Code
1,state_name,str,State
2,district_code,int64,District_Code
3,district_name,str,District
4,lgd_code,int64,(not used / identifier)
5,district_name_lgd,str,(not used / identifier)
6,pop_total,int64,Total_Population
7,pop_male,int64,Male_Population
8,pop_female,int64,Female_Population
9,pop_under6_total,int64,Child_Population


,Conceptual variable,Source column,Present,Dtype
0,State,state_name,True,str
1,State_Code,state_code,True,int64
2,District,district_name,True,str
3,District_Code,district_code,True,int64
4,Total_Population,pop_total,True,int64
5,Male_Population,pop_male,True,int64
6,Female_Population,pop_female,True,int64
7,Child_Population,pop_under6_total,True,int64
8,SC_Population,pop_sc,True,int64
9,ST_Population,pop_st,True,int64


## 3. Data Cleaning and Feature Engineering
`clean_data` keeps the raw DataFrame untouched, works on a copy, coerces numerics, checks totals/denominators/percent ranges, builds unique district labels (names repeat across states, so `district_code` is the key, never names), excludes only non-positive populations, and creates derived variables.

Definitions verified from the data: child population = **age 0-6**; sex ratio = **females per 1000 males**; workforce participation = **workers / total population**. The code below tests which denominator the dataset's `literacy_rate` uses.

In [9]:
# Cell 9 - clean (default: source literacy_rate) and show the quality report
raw_before = raw.copy()
df, quality = pp.clean_data(raw, literacy_basis="source")
assert raw.equals(raw_before), "raw data was modified"
for k, v in quality.items():
    if k != "notes": print(f"{k}: {v}")
print()
for n in quality["notes"]: print("NOTE:", n)

missing_source_columns: []


missing_values: {}
duplicate_rows: 0
duplicate_district_codes: 0
labels_made_unique: 0
invalid_counts: {'non_positive_population': 0, 'negative_counts': 0, 'child_pop_exceeds_total': 0, 'sc_st_exceed_total': 0, 'male_plus_female_ne_total': 0, 'literate_plus_illiterate_ne_total': 0, 'workers_plus_nonworkers_ne_total': 0, 'rates_outside_0_100': 0}
denominator_checks: {'literacy_rate vs literate/pop_total (median abs diff, pp)': 0.0025340710383048304, 'literacy_rate vs literate/(pop_total-under6) (median abs diff, pp)': 9.853240384301813, 'workforce_participation vs workers/pop_total (median abs diff, pp)': 0.002562493787214848, 'sex_ratio vs female/male*1000 (median abs diff)': 0.2630299194891563}
district_name_vs_lgd_mismatch: 90
district_names_repeated_across_states: 12
excluded_rows: 0
literacy_basis: source
rows_after_cleaning: 640

NOTE: The source literacy_rate matches literate_total / TOTAL population (all ages), i.e. a CRUDE literacy rate, not the Census 'effective' rate (age 7+

In [10]:
# Cell 10 - derived variables and formulas
display(pp.derived_variable_table())

,Derived variable,Definition
0,Child_Population_Share,pop_under6_total / pop_total * 100 (child = ag...
1,SC_Population_Share,pop_sc / pop_total * 100
2,ST_Population_Share,pop_st / pop_total * 100
3,Literacy_Rate,Analysis literacy rate. 'source' = dataset col...
4,Literacy_Rate_Source,dataset column literacy_rate (unchanged)
5,Literacy_Rate_Crude_Check,literate_total / pop_total * 100 (used to veri...
6,Literacy_Rate_Effective,literate_total / (pop_total - pop_under6_total...
7,Literacy_Gap,100 - Literacy_Rate
8,Sex_Ratio,dataset column sex_ratio (females per 1000 mal...
9,Workforce_Participation_Rate,dataset column workforce_participation (verifi...


In [11]:
# Cell 11 - schema report: which modules can run
display(pp.module_availability(raw))

,Module,All required fields present
0,A: Population analysis,True
1,B: Education & social,True
2,C: Literacy classification,True
3,D: UIPS,True


> **Literacy definition caveat.** If the check above shows `literacy_rate` matches `literate_total / pop_total`, the dataset's rate is a *crude* all-ages rate. Children aged 0-6 are mostly illiterate by definition, so child share mechanically lowers it. The notebook uses the source `literacy_rate` as required; `pp.clean_data(raw, "effective")` gives the age 7+ rate for comparison (Cell 12).

In [12]:
# Cell 12 - how different are the two literacy bases?
cmp = df[["Literacy_Rate_Source", "Literacy_Rate_Effective", "Child_Population_Share"]]
display(cmp.describe().T)
print(cmp.corr(method="spearman").round(3))

,count,mean,std,min,25%,50%,75%,max
Literacy_Rate_Source,640.0,62.463406,10.527806,28.770000,55.130000,62.030000,70.487500,88.740000
Literacy_Rate_Effective,640.0,72.308429,10.488746,36.099876,65.097558,72.169262,79.962815,97.911499
Child_Population_Share,640.0,13.903929,2.934675,7.546055,11.371201,13.709667,16.067975,23.008220


                         Literacy_Rate_Source  Literacy_Rate_Effective  Child_Population_Share
Literacy_Rate_Source                    1.000                    0.988                  -0.783
Literacy_Rate_Effective                 0.988                    1.000                  -0.686
Child_Population_Share                 -0.783                   -0.686                   1.000


# MODULE A - Population Pressure Analysis
Descriptive analysis of population and child population (age 0-6). No density (no land area in the dataset) and no choropleth (no boundary geometry in the dataset) are produced.

In [13]:
# Cell 13 - descriptive statistics
display(A.population_summary(df).style.format("{:,.2f}"))
print("Total population across included districts:", f"{df['Total_Population'].sum():,.0f}")
print("Concentration:", A.concentration(df, 10))

,Total_Population,Child_Population,Child_Population_Share
count,640.00,640.00,640.00
sum,"1,210,854,977.00","164,515,253.00","8,898.51"
mean,"1,891,960.90","257,055.08",13.90
median,"1,557,367.00","210,436.00",13.71
std,"1,544,380.29","205,756.64",2.93
min,"8,004.00","1,084.00",7.55
Q1,"817,861.00","110,012.75",11.37
Q3,"2,583,551.25","344,275.25",16.07
IQR,"1,765,690.25","234,262.50",4.70
max,"11,060,148.00","1,327,146.00",23.01


Total population across included districts: 1,210,854,977
Concentration: {'top_k': 10, 'top_k_share_%': np.float64(7.127326198370988), 'total_population': np.int64(1210854977)}


In [14]:
# Cell 14 - rankings and state totals
display(A.top_n(df, "Total_Population", 10)); display(A.top_n(df, "Total_Population", 10, ascending=True))
state_pop = A.state_population(df); display(state_pop)

,District_Label,state_name,Total_Population
0,Thane (Maharashtra),Maharashtra,11060148
1,North Twenty Four Parganas (West Bengal),West Bengal,10009781
2,Bangalore (Karnataka),Karnataka,9621551
3,Pune (Maharashtra),Maharashtra,9429408
4,Mumbai Suburban (Maharashtra),Maharashtra,9356962
5,South Twenty Four Parganas (West Bengal),West Bengal,8161961
6,Barddhaman (West Bengal),West Bengal,7717563
7,Ahmadabad (Gujarat),Gujarat,7214225
8,Murshidabad (West Bengal),West Bengal,7103807
9,Jaipur (Rajasthan),Rajasthan,6626178


,District_Label,state_name,Total_Population
0,Dibang Valley (Arunachal Pradesh),Arunachal Pradesh,8004
1,Anjaw (Arunachal Pradesh),Arunachal Pradesh,21167
2,Lahul & Spiti (Himachal Pradesh),Himachal Pradesh,31564
3,Upper Siang (Arunachal Pradesh),Arunachal Pradesh,35320
4,Nicobars (Andaman & Nicobar Islands),Andaman & Nicobar Islands,36842
5,Mahe (Puducherry),Puducherry,41816
6,North District (Sikkim),Sikkim,43709
7,Tawang (Arunachal Pradesh),Arunachal Pradesh,49977
8,Longleng (Nagaland),Nagaland,50484
9,Diu (Daman & Diu),Daman & Diu,52074


,state_name,Districts,Total_Population,Child_Population,Child_Population_Share,Share_of_Included_Population_%
0,Uttar Pradesh,71,199812341,30791331,15.410125,16.501757
1,Maharashtra,35,112374333,13326517,11.859040,9.280577
2,Bihar,38,104099452,19133964,18.380466,8.597186
3,West Bengal,19,91276115,10581466,11.592809,7.538154
4,Andhra Pradesh,23,84580777,9142802,10.809551,6.985211
5,Madhya Pradesh,50,72626809,10809395,14.883478,5.997977
6,Tamil Nadu,32,72147030,7423832,10.289865,5.958354
7,Rajasthan,33,68548437,10649504,15.535736,5.661160
8,Karnataka,30,61095297,7161033,11.721087,5.045633
9,Gujarat,26,60439692,7777262,12.867805,4.991489


In [15]:
# Cell 15 - Module A figures
V.bar_h(A.top_n(df, "Total_Population"), "Total_Population", title="Top 10 districts by total population").show()
V.histogram(df, "Total_Population").show()
V.box(df, "Total_Population").show()
V.state_bar(state_pop, "Total_Population", title="Total population by state (sum of districts)").show()
V.bar_h(A.top_n(df, "Child_Population"), "Child_Population", title="Top 10 districts by child population (age 0-6)").show()
V.scatter(df, "Total_Population", "Child_Population_Share", color="state_name").show()

In [16]:
# Cell 16 - data-driven interpretation (computed, not assumed)
c = A.concentration(df, 10); s = A.population_summary(df)["Total_Population"]
print(f"The 10 most populous districts hold {c['top_k_share_%']:.1f}% of the included population.")
print(f"Mean/median district population = {s['mean']:,.0f} / {s['median']:,.0f} (mean>median indicates right skew: {s['mean']>s['median']}).")
big = A.top_n(df, "Total_Population", 1).iloc[0]
print(f"Largest district: {big['District_Label']}. Count vs share: large child COUNTS follow large populations, whereas child SHARE (scatter) shows demographic structure.")
print("Large population does not mean an urban district, and population alone does not establish an infrastructure deficit.")

The 10 most populous districts hold 7.1% of the included population.
Mean/median district population = 1,891,961 / 1,557,367 (mean>median indicates right skew: True).
Largest district: Thane (Maharashtra). Count vs share: large child COUNTS follow large populations, whereas child SHARE (scatter) shows demographic structure.
Large population does not mean an urban district, and population alone does not establish an infrastructure deficit.


**Interpretation guidance.** Population is concentrated in a minority of districts (see Cell 16). Population *count* indicates aggregate service demand; population *share* (e.g. child share) indicates age structure. Neither shows capacity or deficits.

# MODULE B - Education and Social Development Planning

In [17]:
# Cell 17 - descriptive statistics
display(B.literacy_summary(df).style.format("{:,.2f}"))

,Literacy_Rate,Sex_Ratio,Child_Population_Share,SC_Population_Share,ST_Population_Share,Workforce_Participation_Rate
count,640.00,640.00,640.00,640.00,640.00,640.00
sum,"39,976.58","605,077.00","8,898.51","9,509.93","11,337.10","26,371.18"
mean,62.46,945.43,13.90,14.86,17.71,41.20
median,62.03,949.00,13.71,15.76,4.18,41.20
std,10.53,60.61,2.93,9.13,27.00,7.03
min,28.77,534.00,7.55,0.00,0.00,25.83
Q1,55.13,906.00,11.37,7.50,0.36,35.58
Q3,70.49,983.00,16.07,20.79,21.71,46.58
IQR,15.36,77.00,4.70,13.29,21.35,11.00
max,88.74,"1,184.00",23.01,50.17,98.58,66.90


In [18]:
# Cell 18 - Module B figures
V.bar_h(B.rank_districts(df, "Literacy_Rate"), "Literacy_Rate", title="Top 10 districts by literacy rate").show()
V.bar_h(B.rank_districts(df, "Literacy_Rate", ascending=True), "Literacy_Rate", title="Bottom 10 districts by literacy rate").show()
V.histogram(df, "Literacy_Rate").show(); V.box(df, "Literacy_Rate").show()
sl = B.state_literacy(df, "source")
V.state_bar(sl, "Weighted_Literacy_Rate", title="State literacy (population-weighted: sum literate / sum population)", xlabel="Literacy rate (%)").show()
display(sl)
V.bar_h(B.rank_districts(df, "Sex_Ratio"), "Sex_Ratio", title="Top 10 districts by sex ratio").show()
V.bar_h(B.rank_districts(df, "Sex_Ratio", ascending=True), "Sex_Ratio", title="Bottom 10 districts by sex ratio").show()
display(B.sc_st_table(df, 15))
V.scatter(df, "SC_Population_Share", "ST_Population_Share", color="state_name", title="SC share vs ST share").show()
V.scatter(df, "Child_Population_Share", "Literacy_Rate", color="state_name", trend=True).show()
V.scatter(df, "Workforce_Participation_Rate", "Literacy_Rate", color="state_name", trend=True).show()

,state_name,Districts,Literate,Population,Unweighted_Mean_District_Literacy,Weighted_Literacy_Rate
0,Kerala,14,28135824,33406061,84.137143,84.223710
1,Lakshadweep,1,52553,64473,81.510000,81.511641
2,Goa,2,1165487,1458545,79.750000,79.907511
3,Daman & Diu,2,188406,243247,75.860000,77.454604
4,Andaman & Nicobar Islands,3,294281,380581,74.390000,77.324144
5,Mizoram,8,848175,1097206,75.038750,77.303168
6,Puducherry,4,957309,1247953,77.915000,76.710341
7,Tripura,4,2804783,3673917,75.187500,76.343124
8,Chandigarh,1,805438,1055450,76.310000,76.312284
9,NCT of Delhi,9,12737767,16787941,76.597778,75.874504


,District_Label,state_name,SC_Population_Share,ST_Population_Share,SC_ST_Share
0,Kurung Kumey (Arunachal Pradesh),Arunachal Pradesh,0.000000,98.575090,98.575090
1,Champhai (Mizoram),Mizoram,0.013519,98.187602,98.201121
2,West Khasi Hills (Meghalaya),Meghalaya,0.043811,97.818813,97.862625
3,Tuensang (Nagaland),Nagaland,0.000000,97.110826,97.110826
4,Zunheboto (Nagaland),Nagaland,0.000000,97.018976,97.018976
5,Serchhip (Mizoram),Mizoram,0.049279,96.846174,96.895453
6,Saiha (Mizoram),Mizoram,0.146711,96.585004,96.731714
7,Kiphire (Nagaland),Nagaland,0.000000,96.520458,96.520458
8,Longleng (Nagaland),Nagaland,0.000000,96.297837,96.297837
9,Phek (Nagaland),Nagaland,0.000000,96.161989,96.161989


### Correlation analysis
Pearson measures linear association and is sensitive to outliers/non-normality; Spearman measures monotonic association on ranks. Both are shown. Correlation is not causation.

In [19]:
# Cell 19 - normality check then correlation matrices
for c in B.B_VARS: print(f"Shapiro p for {c}: {ST.shapiro(df[c]):.4g}")
V.corr_heatmap(ST.correlation_matrix(df, B.B_VARS, "pearson"), "Pearson correlation").show()
V.corr_heatmap(ST.correlation_matrix(df, B.B_VARS, "spearman"), "Spearman correlation").show()
display(ST.pair_correlation(df, "Child_Population_Share", "Literacy_Rate"))
display(ST.pair_correlation(df, "Workforce_Participation_Rate", "Literacy_Rate"))

Shapiro p for Literacy_Rate: 0.1058
Shapiro p for Sex_Ratio: 2.992e-12
Shapiro p for Child_Population_Share: 3.35e-08
Shapiro p for SC_Population_Share: 1.07e-09
Shapiro p for ST_Population_Share: 9.561e-33
Shapiro p for Workforce_Participation_Rate: 2.285e-06


,Method,Coefficient,P-value,n
0,Pearson,-0.776495,4.387509e-130,640
1,Spearman,-0.782953,1.200714e-133,640


,Method,Coefficient,P-value,n
0,Pearson,0.004403,0.911471,640
1,Spearman,0.021774,0.582429,640


### Hypothesis test
**RQ:** Do districts with an above-median SC+ST population share differ in literacy rate from other districts?  
**H0:** Literacy-rate distributions are equal in both groups. **H1:** They differ (two-sided). **alpha = 0.05.**  
Test choice: Welch t-test if both groups pass Shapiro-Wilk, otherwise Mann-Whitney U.

In [20]:
# Cell 20 - run the test
h = B.default_hypothesis(df)
for k in ["test", "n", "medians", "statistic", "p_value", "alpha", "effect_size_name", "effect_size", "decision"]: print(f"{k}: {h[k]}")
print(h["interpretation"])

test: Mann-Whitney U
n: (320, 320)
medians: (np.float64(60.519999999999996), np.float64(64.36000000000001))
statistic: 41660.0
p_value: 4.525936203550203e-05
alpha: 0.05
effect_size_name: rank-biserial r
effect_size: 0.18632812499999996
decision: reject H0
Mann-Whitney U: reject H0 at alpha=0.05 (p=4.526e-05, rank-biserial r=0.186). This indicates an association between groups, not causation.


### One-way ANOVA across states
Groups are **states with at least 5 districts** (small states are excluded and listed). Assumptions: independence (only approximately true; districts nest within states), residual normality (Shapiro), equal variances (Levene). If violated, Kruskal-Wallis is the primary result; Welch's ANOVA is also reported.

In [21]:
# Cell 21 - ANOVA with assumption checks
a = B.state_anova(df, 5)
for k, v in a.items(): print(f"{k}: {v}")

groups_used: 26
groups_excluded_small: ['Andaman & Nicobar Islands', 'Chandigarh', 'Dadra & Nagar Haveli', 'Daman & Diu', 'Goa', 'Lakshadweep', 'Puducherry', 'Sikkim', 'Tripura']
n: 618
residual_shapiro_p: 4.5377404838729966e-07
levene_p: 0.00013041213780833307
classic_F: 25.019200040619243
classic_p: 2.4435074118150685e-76
eta_squared: 0.5137497129265185
welch_F: 51.52042194124146
welch_p: 2.598715443761507e-60
kruskal_H: 321.67469242333095
kruskal_p: 2.620905621911438e-53
epsilon_squared: 0.5011396831475186
primary_test: Kruskal-Wallis (assumptions violated)
primary_p: 2.620905621911438e-53
independence_note: Districts are nested within states and may be spatially clustered, so the independence assumption is only approximately satisfied; treat p-values cautiously.
interpretation: Primary: Kruskal-Wallis (assumptions violated), p=2.621e-53 (alpha=0.05). A significant result shows that at least one group differs, not which one, and not why.


In [22]:
# Cell 22 - interpretation (computed)
cc = ST.pair_correlation(df, "Child_Population_Share", "Literacy_Rate").set_index("Method")["Coefficient"]
print(f"Child share vs literacy: Pearson {cc['Pearson']:.3f}, Spearman {cc['Spearman']:.3f}.")
print("With the source (crude) literacy definition this link is partly mechanical (children aged 0-6 count in the denominator).")
print("Associations are not causal. Low literacy does not by itself mean a district lacks schools or needs new ones.")

Child share vs literacy: Pearson -0.776, Spearman -0.783.
With the source (crude) literacy definition this link is partly mechanical (children aged 0-6 count in the denominator).
Associations are not causal. Low literacy does not by itself mean a district lacks schools or needs new ones.


# MODULE C - Literacy Level Classification (Random Forest)
**Target** `Literacy_Category` from the verified overall `Literacy_Rate`: Low < 60%; Medium 60% to < 75%; High >= 75% (project-defined thresholds, not official).  
**Features:** `Sex_Ratio`, `Child_Population_Share`, `Workforce_Participation_Rate`, `SC_Population_Share`, `ST_Population_Share`.  
**Leakage prevention:** literacy rate, male/female literacy, literate/illiterate counts, literacy-derived variables, the target, identifiers and names are excluded (the code asserts this). Imputation is fitted inside a scikit-learn Pipeline on training data only.

In [23]:
# Cell 23 - class distribution and feasibility of a stratified split
dist = C.class_distribution(df, 60, 75); display(dist)
print("Features used:", C.available_features(df))
print("Smallest class:", int(dist["Count"].min()))
print("Spearman of each predictor with Literacy_Rate (diagnostic only):"); display(C.correlation_with_literacy_note(df).round(3))

,Count,Percent,Missing_target
Low,261,40.78125,0
Medium,301,47.03125,0
High,78,12.18750,0


Features used: ['Sex_Ratio', 'Child_Population_Share', 'Workforce_Participation_Rate', 'SC_Population_Share', 'ST_Population_Share', 'Child_Sex_Ratio', 'Female_Workforce_Rate', 'Dependency_Ratio', 'Log_Population']
Smallest class: 78
Spearman of each predictor with Literacy_Rate (diagnostic only):


Sex_Ratio                       0.137
Child_Population_Share         -0.783
Workforce_Participation_Rate    0.022
SC_Population_Share             0.045
ST_Population_Share            -0.183
Child_Sex_Ratio                -0.058
Female_Workforce_Rate          -0.159
Dependency_Ratio               -0.126
Log_Population                  0.002
Name: Literacy_Rate, dtype: float64

In [24]:
# Cell 24 - train baseline + Random Forest (80:20, stratified when possible, fixed seed)
res = C.train_evaluate(df, low=60, high=75, test_size=0.2, seed=SEED)
for w in res["warnings"]: print("WARNING:", w)
print("train/test:", res["n_train"], res["n_test"], "| stratified:", res["stratified"], "| missing-target exclusions:", res["n_missing_target"])
comp = pd.DataFrame({m: {k: res[m2][k] for k in ["accuracy", "balanced_accuracy", "macro_f1"]} for m, m2 in [("Baseline", "baseline"), ("Random Forest", "rf")]}).T
display(comp.round(4))
if res["cv_macro_f1"] is not None: print("Training-set CV macro-F1: %.3f +/- %.3f" % (res["cv_macro_f1"].mean(), res["cv_macro_f1"].std()))

train/test: 512 128 | stratified: True | missing-target exclusions: 0


,accuracy,balanced_accuracy,macro_f1
Baseline,0.4688,0.3333,0.2128
Random Forest,0.7422,0.6375,0.6606


Training-set CV macro-F1: 0.705 +/- 0.045


In [25]:
# Cell 25 - per-class metrics, report, confusion matrix, feature importance
display(res["rf"]["per_class"].round(3)); print(res["rf"]["report"])
display(res["rf"]["confusion"])
V.confusion_heatmap(res["rf"]["confusion"]).show()
V.importance_bar(res["importance"]).show(); display(res["importance"].round(4))
display(res["test_table"].head(20))

,Precision,Recall,F1,Support
Low,0.812,0.750,0.780,52
Medium,0.699,0.850,0.767,60
High,0.714,0.312,0.435,16


              precision    recall  f1-score   support

         Low       0.81      0.75      0.78        52
      Medium       0.70      0.85      0.77        60
        High       0.71      0.31      0.43        16

    accuracy                           0.74       128
   macro avg       0.74      0.64      0.66       128
weighted avg       0.75      0.74      0.73       128



,Pred Low,Pred Medium,Pred High
True Low,39,13,0
True Medium,7,51,2
True High,2,9,5


,Feature,Impurity_Importance,Permutation_Importance
0,Child_Population_Share,0.4064,0.2512
1,Sex_Ratio,0.1027,-0.0111
2,Female_Workforce_Rate,0.0849,0.0276
3,SC_Population_Share,0.0830,0.0141
4,ST_Population_Share,0.0710,0.0227
5,Log_Population,0.0679,0.0137
6,Child_Sex_Ratio,0.0671,0.0153
7,Dependency_Ratio,0.0660,-0.0033
8,Workforce_Participation_Rate,0.0508,0.0013


,District,Actual,Predicted,Predicted_RF,Predicted_Baseline,Correct
0,Cuddalore (Tamil Nadu),Medium,Medium,Medium,Medium,True
1,Rewa (Madhya Pradesh),Medium,Medium,Medium,Medium,True
2,Sitapur (Uttar Pradesh),Low,Low,Low,Medium,True
3,Morigaon (Assam),Low,Low,Low,Medium,True
4,Jabalpur (Madhya Pradesh),Medium,Medium,Medium,Medium,True
5,Jagatsinghapur (Odisha),High,High,High,Medium,True
6,Saiha (Mizoram),Medium,Low,Low,Medium,False
7,Janjgir - Champa (Chhattisgarh),Medium,Low,Low,Medium,False
8,Jaunpur (Uttar Pradesh),Medium,Medium,Medium,Medium,True
9,West Tripura (Tripura),High,Medium,Medium,Medium,False


In [26]:
# Cell 26 - computed interpretation
pc = res["rf"]["per_class"]; cm = res["rf"]["confusion"].to_numpy(); labs = res["labels"]
print("Hardest class (lowest F1):", pc["F1"].idxmin(), f"(F1={pc['F1'].min():.3f})")
off = [(labs[i], labs[j], cm[i, j]) for i in range(len(labs)) for j in range(len(labs)) if i != j and cm[i, j] > 0]
print("Most frequent confusion (actual -> predicted):", max(off, key=lambda t: t[2]) if off else "none")
print("Top features:", res["importance"]["Feature"].head(3).tolist())
d = res["rf"]["macro_f1"] - res["baseline"]["macro_f1"]
print(f"Macro-F1 improvement over baseline: {d:+.3f}")
print("The model finds demographic associations within Census 2011 only; it does not predict future literacy, prove causation, or verify school shortages.")

Hardest class (lowest F1): High (F1=0.435)
Most frequent confusion (actual -> predicted): ('Low', 'Medium', np.int64(13))
Top features: ['Child_Population_Share', 'Sex_Ratio', 'Female_Workforce_Rate']
Macro-F1 improvement over baseline: +0.448
The model finds demographic associations within Census 2011 only; it does not predict future literacy, prove causation, or verify school shortages.


# MODULE D - District Demographic Infrastructure Planning Priority Score (UIPS)
The dataset has no urban-specific indicators, so UIPS is a **district-level demographic planning priority score**, not a measure of urban infrastructure deficit.

`UIPS = 100 x (0.35 P + 0.25 C + 0.25 L + 0.15 W)` where P = normalised total population, C = normalised child share, L = normalised literacy gap (100 - literacy rate), W = normalised workforce participation. Min-max: `(x - min) / (max - min)`; where lower values raise priority: `1 - normalised`.

**Workforce direction (documented assumption):** lower participation is treated as higher priority. This is a weak justification and overlaps with child share (children are non-workers), so Cell 29 checks component correlations and the sensitivity analysis includes a no-workforce scenario with weights renormalised (0.35/0.85, 0.25/0.85, 0.25/0.85 = 41.18%, 29.41%, 29.41%). Weights are provisional assumptions, not planning standards. Module C predictions are not used here (no literacy double counting).

In [27]:
# Cell 27 - compute UIPS (default four components)
print(D.WORKFORCE_RATIONALE)
tab, info = D.compute_uips(df)
print("weights:", {k: round(v, 4) for k, v in info["weights"].items()}, "| sum =", round(info["weight_sum"], 6))
print("excluded (missing component):", info["excluded_missing"], "| warnings:", info["warnings"])
print("UIPS range:", tab["UIPS"].min().round(2), "to", tab["UIPS"].max().round(2), "| non-finite:", int((~np.isfinite(tab["UIPS"].dropna())).sum()))
display(tab.head(15))

Assumption: lower workforce participation (workers / total population) is treated as higher priority because it may indicate a weaker local employment base and higher dependency burden. This is a weak, assumption-laden direction (it overlaps with child share, since children are non-workers); it can be switched or omitted.
weights: {'Total_Population': 0.35, 'Child_Population_Share': 0.25, 'Literacy_Gap': 0.25, 'Workforce_Participation_Rate': 0.15} | sum = 1.0
excluded (missing component): 0 | warnings: []
UIPS range: 10.8 to 66.58 | non-finite: 0


,District_Label,state_name,Total_Population,Child_Population_Share,Literacy_Gap,Workforce_Participation_Rate,P_norm,C_norm,L_norm,W_norm,UIPS,P_contribution,C_contribution,L_contribution,W_contribution,Priority_Category,Rank
0,Purba Champaran (Bihar),Bihar,5099371,19.969071,55.35,34.07,0.460668,0.803446,0.735201,0.799367,66.580052,16.123373,20.086151,18.380023,11.990504,Medium,1.0
1,Mewat (Haryana),Haryana,1089263,22.779439,58.24,26.62,0.097833,0.985204,0.783392,0.980765,62.350494,3.424138,24.630095,19.584792,14.711468,Medium,2.0
2,Purnia (Bihar),Bihar,3264619,20.109850,59.19,35.02,0.294659,0.812551,0.799233,0.776236,62.251200,10.313069,20.313771,19.980824,11.643535,Medium,3.0
3,Katihar (Bihar),Bihar,3071029,20.199484,58.31,33.00,0.277143,0.818348,0.784559,0.825420,62.153976,9.700007,20.458695,19.613974,12.381300,Medium,4.0
4,Sitamarhi (Bihar),Bihar,3423574,19.372358,58.03,32.56,0.309041,0.764854,0.779890,0.836133,61.977055,10.816449,19.121356,19.497249,12.542001,Medium,5.0
5,Bahraich (Uttar Pradesh),Uttar Pradesh,3487731,18.774212,59.91,33.03,0.314846,0.726170,0.811239,0.824690,61.825183,11.019622,18.154244,20.280974,12.370343,Medium,6.0
6,Budaun (Uttar Pradesh),Uttar Pradesh,3681896,18.058875,57.97,30.08,0.332414,0.679906,0.778889,0.896518,61.552165,11.634505,16.997652,19.472236,13.447772,Medium,7.0
7,Pashchim Champaran (Bihar),Bihar,3935042,19.721035,55.29,37.58,0.355319,0.787405,0.734200,0.713903,61.184841,12.436169,19.685115,18.355011,10.708546,Medium,8.0
8,Darbhanga (Bihar),Bihar,3937385,18.425478,53.86,31.08,0.355531,0.703616,0.710355,0.872169,60.875404,12.443589,17.590394,17.758879,13.082542,Medium,9.0
9,Samastipur (Bihar),Bihar,4261566,18.710986,49.72,31.52,0.384863,0.722081,0.641321,0.861456,60.477081,13.470207,18.052017,16.033017,12.921841,Medium,10.0


In [28]:
# Cell 28 - Module D figures and district table
V.bar_h(tab.head(10), "UIPS", title="Top 10 districts by UIPS").show()
V.bar_h(tab.dropna(subset=["UIPS"]).tail(10), "UIPS", title="Bottom 10 districts by UIPS").show()
V.histogram(tab, "UIPS").show(); V.box(tab, "UIPS").show(); V.category_bar(tab).show()
full = tab.drop(columns=[c for c in ("Total_Population", "Literacy_Gap") if c in tab]).merge(df[["District_Label", "Total_Population", "Literacy_Gap"]], on="District_Label", how="left")
V.scatter(full, "Total_Population", "UIPS", title="Total population vs UIPS").show()
V.scatter(full, "Literacy_Gap", "UIPS", title="Literacy gap vs UIPS").show()
display(tab)

,District_Label,state_name,Total_Population,Child_Population_Share,Literacy_Gap,Workforce_Participation_Rate,P_norm,C_norm,L_norm,W_norm,UIPS,P_contribution,C_contribution,L_contribution,W_contribution,Priority_Category,Rank
0,Purba Champaran (Bihar),Bihar,5099371,19.969071,55.35,34.07,0.460668,0.803446,0.735201,0.799367,66.580052,16.123373,20.086151,18.380023,11.990504,Medium,1.0
1,Mewat (Haryana),Haryana,1089263,22.779439,58.24,26.62,0.097833,0.985204,0.783392,0.980765,62.350494,3.424138,24.630095,19.584792,14.711468,Medium,2.0
2,Purnia (Bihar),Bihar,3264619,20.109850,59.19,35.02,0.294659,0.812551,0.799233,0.776236,62.251200,10.313069,20.313771,19.980824,11.643535,Medium,3.0
3,Katihar (Bihar),Bihar,3071029,20.199484,58.31,33.00,0.277143,0.818348,0.784559,0.825420,62.153976,9.700007,20.458695,19.613974,12.381300,Medium,4.0
4,Sitamarhi (Bihar),Bihar,3423574,19.372358,58.03,32.56,0.309041,0.764854,0.779890,0.836133,61.977055,10.816449,19.121356,19.497249,12.542001,Medium,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,Leh(Ladakh) (Jammu & Kashmir),Jammu & Kashmir,133487,9.001626,29.75,56.24,0.011354,0.094138,0.308321,0.259557,14.352193,0.397380,2.353439,7.708021,3.893353,Low,636.0
636,Daman (Daman & Diu),Daman & Diu,191173,10.701302,21.36,55.20,0.016573,0.204062,0.168418,0.284879,14.165252,0.580061,5.101561,4.210439,4.273192,Low,637.0
637,Lahul & Spiti (Himachal Pradesh),Himachal Pradesh,31564,9.900520,30.79,61.13,0.002132,0.152273,0.325663,0.140492,14.130374,0.074610,3.806816,8.141571,2.107378,Low,638.0
638,Mokokchung (Nagaland),Nagaland,194622,10.551736,18.05,51.42,0.016885,0.194389,0.113223,0.376917,13.935062,0.590983,4.859735,2.830582,5.653762,Low,639.0


In [29]:
# Cell 29 - double-counting check and which components drive the top districts
display(D.component_correlations(df).round(2))
display(D.contribution_summary(tab, 10).round(2))
print("Categories:", tab["Priority_Category"].value_counts().to_dict())

,Total_Population,Child_Population_Share,Literacy_Gap,Workforce_Participation_Rate
Total_Population,1.00,-0.11,-0.00,-0.27
Child_Population_Share,-0.11,1.00,0.78,-0.24
Literacy_Gap,-0.00,0.78,1.00,-0.02
Workforce_Participation_Rate,-0.27,-0.24,-0.02,1.00


,Rank,District_Label,state_name,UIPS,P_contribution,C_contribution,L_contribution,W_contribution
0,1.0,Purba Champaran (Bihar),Bihar,66.58,16.12,20.09,18.38,11.99
1,2.0,Mewat (Haryana),Haryana,62.35,3.42,24.63,19.58,14.71
2,3.0,Purnia (Bihar),Bihar,62.25,10.31,20.31,19.98,11.64
3,4.0,Katihar (Bihar),Bihar,62.15,9.70,20.46,19.61,12.38
4,5.0,Sitamarhi (Bihar),Bihar,61.98,10.82,19.12,19.50,12.54
5,6.0,Bahraich (Uttar Pradesh),Uttar Pradesh,61.83,11.02,18.15,20.28,12.37
6,7.0,Budaun (Uttar Pradesh),Uttar Pradesh,61.55,11.63,17.00,19.47,13.45
7,8.0,Pashchim Champaran (Bihar),Bihar,61.18,12.44,19.69,18.36,10.71
8,9.0,Darbhanga (Bihar),Bihar,60.88,12.44,17.59,17.76,13.08
9,10.0,Samastipur (Bihar),Bihar,60.48,13.47,18.05,16.03,12.92


Categories: {'Medium': 384, 'Low': 256}


### Sensitivity analysis
Five weighting scenarios (each sums to 1): baseline; equal weights; population-heavy; social-development-heavy; without workforce (renormalised). No single weight set is uniquely correct.

In [30]:
# Cell 30 - sensitivity analysis
S_, R_, C_, summ, rank_corr, changed = D.sensitivity_analysis(df)
display(summ.round(3)); display(rank_corr.round(3))
print(f"{len(changed)} districts change category in at least one scenario"); display(changed.head(30))
for n, w in D.SCENARIOS.items(): print(n, "sum of weights =", round(sum(D.normalize_weights(w).values()), 6))

,Scenario,Weights,Mean_abs_rank_change_vs_base,Max_abs_rank_change_vs_base,Top10_overlap_with_base,Spearman_vs_base,Districts_changing_category
0,Baseline (illustrative),"{'Total_Population': 0.35, 'Child_Population_S...",0.000,0.0,10,1.000,0
1,Equal weights,"{'Total_Population': 0.25, 'Child_Population_S...",24.916,124.0,9,0.984,103
2,Population-heavy,"{'Total_Population': 0.5, 'Child_Population_Sh...",45.300,211.0,1,0.947,114
3,Social-development-heavy,"{'Total_Population': 0.2, 'Child_Population_Sh...",35.766,284.0,7,0.963,83
4,Without workforce (renormalised),"{'Total_Population': 0.4118, 'Child_Population...",34.700,174.0,7,0.972,89


,Baseline (illustrative),Equal weights,Population-heavy,Social-development-heavy,Without workforce (renormalised)
Baseline (illustrative),1.000,0.984,0.947,0.963,0.972
Equal weights,0.984,1.000,0.909,0.966,0.925
Population-heavy,0.947,0.909,1.000,0.834,0.901
Social-development-heavy,0.963,0.966,0.834,1.000,0.953
Without workforce (renormalised),0.972,0.925,0.901,0.953,1.000


257 districts change category in at least one scenario


,Baseline (illustrative),Equal weights,Population-heavy,Social-development-heavy,Without workforce (renormalised)
District_Label,,,,,
Alirajpur (Madhya Pradesh),Medium,Medium,Medium,High,Medium
Ambala (Haryana),Low,Medium,Low,Low,Low
Amreli (Gujarat),Low,Medium,Low,Low,Low
Amritsar (Punjab),Medium,Medium,Low,Medium,Low
Anand (Gujarat),Low,Medium,Low,Low,Low
Anjaw (Arunachal Pradesh),Medium,Medium,Low,Medium,Medium
Anugul (Odisha),Low,Medium,Low,Low,Low
Anuppur (Madhya Pradesh),Low,Medium,Low,Medium,Low
Araria (Bihar),Medium,Medium,Medium,High,Medium


Baseline (illustrative) sum of weights = 1.0
Equal weights sum of weights = 1.0
Population-heavy sum of weights = 1.0
Social-development-heavy sum of weights = 1.0
Without workforce (renormalised) sum of weights = 1.0


In [31]:
# Cell 31 - final interpretation (computed)
top = tab.head(5)
print("Highest UIPS districts:", top["District_Label"].tolist())
contrib = [c for c in tab.columns if c.endswith("_contribution")]
print("Average component contributions among the top 10 (percentage points):"); print(tab.head(10)[contrib].mean().round(2))
print()
print("Ranks are relative to the included districts and depend on indicators and weights (see Cell 30).")
print("UIPS = demographic planning priority, NOT a verified infrastructure shortage, and NOT a justification to build new infrastructure.")
print("Census 2011 does not measure current school capacity, classroom crowding, roads, hospital capacity, housing or service availability.")

Highest UIPS districts: ['Purba Champaran (Bihar)', 'Mewat (Haryana)', 'Purnia (Bihar)', 'Katihar (Bihar)', 'Sitamarhi (Bihar)']
Average component contributions among the top 10 (percentage points):
P_contribution    11.14
C_contribution    19.51
L_contribution    18.90
W_contribution    12.58
dtype: float64

Ranks are relative to the included districts and depend on indicators and weights (see Cell 30).
UIPS = demographic planning priority, NOT a verified infrastructure shortage, and NOT a justification to build new infrastructure.
Census 2011 does not measure current school capacity, classroom crowding, roads, hospital capacity, housing or service availability.


## 12. Validation checklist (run in your environment)
Re-run the notebook top to bottom after loading the real dataset; every printed number above comes from the data. Results are not pre-filled.

In [32]:
# Cell 32 - final self-checks
assert not set(res["features"]) & C.FORBIDDEN, "leakage"
for n, w in D.SCENARIOS.items(): assert abs(sum(D.normalize_weights(w).values()) - 1) < 1e-9
assert tab["UIPS"].dropna().between(0, 100).all()
print("All self-checks passed.")

All self-checks passed.
